# N1 — Data Integrity

## Decision question

Is the available information reliable enough for a liquidity decision, and
which assumptions still require human confirmation?

### Learning objectives

- Separate blocking integrity failures from business-rule warnings.
- Explore the size, concentration, timing, and operating context of the case.
- Build an assumptions register rather than hiding illustrative inputs.


In [ ]:
from pathlib import Path
import json
import sys

# Find the public package locally. A fresh Colab runtime downloads the same
# participant-safe assets from the repository.
for candidate in [Path.cwd(), *Path.cwd().parents]:
    for source_candidate in (candidate / 'src', candidate / 'CFOPackV002' / 'src'):
        if (source_candidate / 'workshop_bootstrap.py').exists():
            sys.path.insert(0, str(source_candidate))
            break

try:
    from workshop_bootstrap import bootstrap
except ImportError:
    from urllib.request import urlopen
    bootstrap_url = (
        'https://raw.githubusercontent.com/VinayaSharada/'
        'KateelLearningDemosToStudents/cfopack-v002-v2.1.0-beta.1/CFOPackV002/src/workshop_bootstrap.py'
    )
    namespace = {}
    exec(compile(urlopen(bootstrap_url).read(), bootstrap_url, 'exec'), namespace)
    bootstrap = namespace['bootstrap']

ROOT, OUTPUT_DIR = bootstrap()
from cfopack_v002 import (
    analyze_fx,
    default_decisions,
    load_inputs,
    load_manifest,
    reveal_team_shock,
    run_pipeline,
)
import workshop_visuals as viz
import pandas as pd
try:
    from IPython.display import Markdown, display
except ImportError:
    # Keep the notebooks runnable from a minimal local Python environment as
    # well as Colab/Jupyter. Rich notebook rendering remains the default.
    def Markdown(value):
        return value

    def display(value):
        print(value)

manifest = load_manifest(ROOT / 'config' / 'scenario_manifest.json')
decision_file = OUTPUT_DIR / 'N0_team_decisions.json'
if decision_file.exists():
    DECISIONS = json.loads(decision_file.read_text(encoding='utf-8'))
else:
    DECISIONS = default_decisions(manifest)


In [ ]:
data = load_inputs(ROOT / 'data' / 'synthetic')
viz.data_snapshot(data, OUTPUT_DIR, 'N1')


In [ ]:
summary = run_pipeline(ROOT, OUTPUT_DIR, DECISIONS)
print(f"Scenario {summary['scenario_version']} calculated for {DECISIONS['team_name']} (model cache: {'hit' if summary['model_cache_hit'] else 'rebuilt'})")


## Explore the decision data


In [ ]:
data = load_inputs(ROOT / 'data' / 'synthetic')
outstanding = data['invoices'].query("status == 'outstanding'")
print(f"Outstanding AR: ${outstanding['amount_usd'].sum():,.0f} across {len(outstanding):,} invoices")
print(f"Historical payments: {len(data['payments']):,}")
print(f"Forecast outflows: ${data['operating_outflows']['total_operating_outflows'].sum():,.0f} operating + "
      f"${data['supplier_payments']['amount_usd'].sum():,.0f} suppliers")
viz.data_landscape(data, OUTPUT_DIR)


## Integrity checks and assumptions


In [ ]:
validation = pd.read_csv(OUTPUT_DIR / 'N1_validation_report.csv')
assumptions = pd.read_csv(OUTPUT_DIR / 'N1_assumptions_register.csv')
display(validation)
display(assumptions)
blocking_failures = validation.query("blocking == True and status == 'FAIL'")
print('Decision-ready' if blocking_failures.empty else 'STOP: resolve blocking failures')
viz.validation_chart(validation, assumptions, OUTPUT_DIR)


## Team decision

Identify the three assumptions most likely to change the CFO recommendation.
For each, name an owner and the evidence required to confirm it.


In [ ]:
# Make the integrity decision operational. Selecting 'rejected' deliberately
# stops downstream analysis until the data issue is resolved.
DECISIONS['data_approval'] = 'conditional'  # approved, conditional, rejected
decision_file.write_text(json.dumps(DECISIONS, indent=2), encoding='utf-8')
print(f"Data approval recorded: {DECISIONS['data_approval']}")
if DECISIONS['data_approval'] == 'rejected':
    raise RuntimeError('Team rejected the data. Resolve the blocking issue before N2.')
summary = run_pipeline(ROOT, OUTPUT_DIR, DECISIONS)


### Before moving on

Record your interpretation in the participant workbook. Do not copy a chart
without also recording the assumption and decision it supports.
